# 05 — SCL-OPP: Opposing-Polarity Phrases

Loads the **SCL-OPP** lexicon — 1,178 phrase-level entries where
the phrase meaning **opposes** a naive reading of its parts
(e.g. `bad luck`, `happy accident`, `pretty ugly`).

**Source:** `data/SCL-OPP/SCL-OPP/SCL-OPP.txt`  
**Scale:** bipolar valence [-1, +1]  
**Columns kept:** all originals (`term`, `opp_valence`, `opp_pos`, `opp_freq`, `opp_is_hashtag`)


In [1]:
from pathlib import Path
import pandas as pd
import plotly.express as px

OPP_FILE = Path('data/SCL-OPP/SCL-OPP/SCL-OPP.txt')


## Load

In [2]:
opp = pd.read_csv(OPP_FILE, sep='\t', header=None,
                  names=['term', 'opp_valence', 'opp_pos', 'opp_freq'])

# Hashtag terms use '#' as a phrase-boundary marker — flag then strip
opp['opp_is_hashtag'] = opp['term'].str.contains('#', regex=False)
opp['term'] = opp['term'].str.replace('#', '', regex=False).str.strip()

print(f'Entries       : {len(opp):,}')
print(f'Hashtag terms : {opp["opp_is_hashtag"].sum():,}')
print(f'Valence range : [{opp["opp_valence"].min():.3f}, {opp["opp_valence"].max():.3f}]')
print(f'Columns       : {list(opp.columns)}')
opp.head(5)


Entries       : 1,178
Hashtag terms : 12
Valence range : [-1.000, 1.000]
Columns       : ['term', 'opp_valence', 'opp_pos', 'opp_freq', 'opp_is_hashtag']


,term,opp_valence,opp_pos,opp_freq,opp_is_hashtag
0,seriously great,1.000,R+A,17,False
1,ridiculously happy,1.000,R+A,33,False
2,amazing,1.000,A,100554,False
3,pretty damn amazing,0.969,R+R+A,112,False
4,happiness overload,0.969,N+N,60,False


## Valence distribution

In [3]:
fig = px.histogram(opp, x='opp_valence', nbins=40,
    title='SCL-OPP valence distribution',
    labels={'opp_valence': 'Valence [-1, +1]'})
fig.add_vline(x=0, line_dash='dash', line_color='grey')
fig.show()

print(opp['opp_valence'].describe().round(4).to_string())
print(f'\nPositive (>0) : {(opp["opp_valence"] > 0).sum():,}')
print(f'Negative (<0) : {(opp["opp_valence"] < 0).sum():,}')
print(f'Neutral  (=0) : {(opp["opp_valence"] == 0).sum():,}')


count    1178.0000
mean       -0.0705
std         0.5064
min        -1.0000
25%        -0.5000
50%        -0.0780
75%         0.3130
max         1.0000

Positive (>0) : 519
Negative (<0) : 647
Neutral  (=0) : 12


## N-gram breakdown

In [4]:
opp['ngrams'] = opp['term'].str.split().str.len()
print(opp['ngrams'].value_counts().sort_index().rename({1:'unigram',2:'bigram',3:'trigram'}).to_string())

fig2 = px.box(opp, x=opp['ngrams'].map({1:'unigram',2:'bigram',3:'trigram'}).fillna('4+'),
    y='opp_valence', points='all',
    title='SCL-OPP valence by n-gram length',
    labels={'x':'n-gram type','opp_valence':'Valence'})
fig2.show()


ngrams
unigram    602
bigram     311
trigram    265


## Most positive / most negative

In [5]:
print('Most positive:')
display(opp.nlargest(10, 'opp_valence')[['term','opp_valence','opp_pos','opp_freq']])
print('\nMost negative:')
display(opp.nsmallest(10, 'opp_valence')[['term','opp_valence','opp_pos','opp_freq']])


Most positive:


,term,opp_valence,opp_pos,opp_freq
0,seriously great,1.000,R+A,17
1,ridiculously happy,1.000,R+A,33
2,amazing,1.000,A,100554
3,pretty damn amazing,0.969,R+R+A,112
4,happiness overload,0.969,N+N,60
5,love,0.969,V,770553
6,so unbelievable happy,0.953,R+A+A,16
7,smiling,0.953,V,10050
8,wonderful,0.953,A,37717
9,smile,0.953,#,57000



Most negative:


,term,opp_valence,opp_pos,opp_freq
1177,sucks to live,-1.000,V+P+V,13
1170,like to torture,-0.984,V+P+V,8
1171,please kill me,-0.984,V+V+O,37
1172,just died,-0.984,R+V,654
1173,feel like crap,-0.984,V+P+N,619
1174,friend died,-0.984,N+V,45
1175,died,-0.984,V,10969
1176,dying,-0.984,V,5626
1168,truly hate,-0.969,R+V,22
1169,disaster,-0.969,N,788
